# Project 1 (solution, open models): An LLM as a coder for qualitative interviews

**Hands-on project, about 2 hours, in teams or alone.**

In December 2025 Anthropic released **1,250 interview transcripts** in which professionals describe how they use AI at work and how they feel about it. The interviews were run by an AI interviewer, and every participant agreed to public release. The data sit on Hugging Face as [`Anthropic/AnthropicInterviewer`](https://huggingface.co/datasets/Anthropic/AnthropicInterviewer) and come in three groups:

| Group | Interviews |
|---|---|
| `workforce` (general workforce) | 1,000 |
| `creatives` (writers, artists, musicians, designers) | 125 |
| `scientists` | 125 |

**Main research questions:** *Do the three occupational groups differ in their attitude towards AI and in what worries them most?*

A qualitative researcher would answer this by reading every transcript and coding it with a codebook. 

**You do not write any Python.** Run the cells top to bottom. Wherever a cell says `TODO`, you make a research decision: write a prompt in plain English, choose a setting, or pick a model. Replace the `...` with your choice and run the cell. If you forget one, the cell stops and tells you which value is missing.


> **Open-model version.** This notebook runs an open-weight model (**Qwen2.5-3B-Instruct** by default) on your own GPU through Hugging Face `transformers`, instead of calling the OpenAI API. No API key is needed, and no text leaves your machine, which matters for confidential material such as interviews. A GPU is needed; a free Colab or Kaggle T4 is enough.

> **Solution notebook (instructor version).** Every `TODO` is filled in with one possible solution, and the discussion questions have sample answers. Model outputs change with the model and its version, so the answers describe what to look for rather than quoting numbers.

## Setup

Run these cells first. The first installs the libraries (needed on Colab or Kaggle), and the second lets Colab display progress bars. The third loads an open-weight model onto the GPU. The fourth defines helpers you do not need to edit:

- `ask()` sends a prompt to the local model and returns the reply. With `json_mode=True` it **prefills** the answer with `{`, so the model has to continue a JSON object instead of starting with "Sure! Here is...". This is the technique from the *Prompting* demo. Its `model=` argument sends the prompt to a different loaded model instead.
- `parse_json()` turns that reply into a Python dict, or `{}` if it is not valid JSON.
- `run_parallel()` codes a list of interviews with a progress bar. A single local model answers one request at a time, so it just loops.
- `require()` stops a cell with a clear message if you have not yet filled in one of its `TODO` values.

**Which model?** `Qwen2.5-3B-Instruct` (about 6 GB) fits a T4 and is the smallest Qwen model that follows a codebook reasonably well. `Qwen2.5-1.5B-Instruct` is faster but noticeably worse at following instructions. With 24 GB or more of GPU memory, `Qwen2.5-7B-Instruct` is also an option. Try them all. 

In [1]:
!pip install -q datasets

In [2]:
from google.colab import output
output.enable_custom_widget_manager()

In [3]:
import re
import json
import warnings

import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import pipeline
from transformers import logging as hf_logging

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()

MODEL = "Qwen/Qwen2.5-3B-Instruct"

pipe = pipeline(
    "text-generation",
    model=MODEL,
    dtype="auto",          # on a Colab T4 use dtype=torch.float16: bfloat16 is emulated there and slow
    device_map="auto",
)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Model : {MODEL} on {pipe.model.device} ({pipe.model.dtype})")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model : Qwen/Qwen2.5-3B-Instruct on cuda:0 (torch.bfloat16)


In [4]:
def ask(system, user, json_mode=False, model=None, **decoding):
    """Send one system + user message pair to a local model and return the reply text.

    decoding: generation parameters passed straight to the pipeline, e.g.
              do_sample=False, max_new_tokens=200, temperature=0.7, top_p=0.9.
    model: another loaded pipeline to use instead of `pipe`, e.g. a second coder.
    """
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    if json_mode:
        # Prefill: the model continues this assistant message, so its reply starts inside the JSON.
        messages.append({"role": "assistant", "content": "{"})

    generator = pipe if model is None else model
    reply = generator(messages, **decoding)[0]["generated_text"][-1]["content"]
    if json_mode and not reply.lstrip().startswith("{"):
        reply = "{" + reply          # some versions return only the continuation
    return reply.strip()


def parse_json(reply):
    """Turn the model's reply into a dict; return {} if it is not valid JSON."""
    start = reply.find("{")
    if start == -1:
        return {}
    try:
        parsed, _ = json.JSONDecoder().raw_decode(reply[start:])   # ignores anything after the closing brace
        return parsed if isinstance(parsed, dict) else {}
    except json.JSONDecodeError:
        return {}


def run_parallel(fn, items, workers=None):
    """Apply fn to every item, in order; a local model runs one at a time."""
    return [fn(item) for item in tqdm(items)]


def require(**values):
    """Stop with a clear message if a TODO value is still ... ."""
    missing = [name for name, value in values.items() if value is ... or value == "..."]
    if missing:
        raise ValueError(f"Fill in the TODO first: {', '.join(missing)}")

## Step 1 - Load the data and read before you code

Each transcript alternates between the interviewer (`AI:` or `Assistant:`) and the participant (`User:`). The interviewer often summarises what the participant said, sometimes more warmly than the participant did. We therefore code **only the participant's answers**, which `participant_text()` extracts.

You choose how many interviews per group to code. We sample the same number from each group because the groups are very unequal in size: a random 60 from all 1,250 would contain only about 6 creatives and 6 scientists. More interviews give more precise proportions but take longer to code. On a T4, allow roughly 5 seconds per interview per model.

In [16]:
TURN = re.compile(r"^(User|AI|Assistant):", re.MULTILINE)


def participant_text(transcript):
    """Keep only what the interviewee said, one answer per paragraph."""
    parts = TURN.split(transcript)
    # parts looks like ['', 'Assistant', ' Hi! ...', 'User', ' Sounds good ...', 'AI', ...]
    answers = [text.strip() for speaker, text in zip(parts[1::2], parts[2::2]) if speaker == "User"]
    return "\n\n".join(answers)


ds = load_dataset("Anthropic/AnthropicInterviewer")
interviews = pd.concat(
    [ds[split].to_pandas().assign(group=split) for split in ds],
    ignore_index=True,
)
interviews["answers"] = interviews["text"].apply(participant_text)

# 60 per group: enough for rough proportions, and both models finish in about 10 minutes on a T4.
N_PER_GROUP = 60
require(N_PER_GROUP=N_PER_GROUP)
sample = (
    interviews.groupby("group")
    .sample(n=N_PER_GROUP, random_state=42)
    .reset_index(drop=True)
)

print(interviews["group"].value_counts(), "\n")
print(f"Sample: {len(sample)} interviews")
print(f"Answer length in the sample: median {sample['answers'].str.len().median():.0f} characters")

group
workforce     1000
creatives      125
scientists     125
Name: count, dtype: int64 

Sample: 180 interviews
Answer length in the sample: median 3698 characters


Now **read**. The next cell prints one interview from each group. Each team member reads one of them and notes, in a few words: *How does this person feel about AI in their work? What worries them most?*

Do this before looking at the codebook in Step 2. The themes you notice yourselves are the ones you will check the codebook against.

In [17]:
# Each team member reads one of the interviews below and writes down, in a few words,
# how the person feels about AI in their work and what worries them most.
for i in [0, N_PER_GROUP, 2 * N_PER_GROUP]:     # one interview per group: creatives, scientists, workforce
    row = sample.iloc[i]
    print("=" * 80)
    print(f"{row.transcript_id}  |  group: {row.group}\n")
    print(row.answers, "\n")

creativity_0018  |  group: creatives

No, I understand.

I'm a musician, I compose music and mix the audio. I use logic pro x as my DAW and things like native instruments plugins for the instruments. A typical project isn't always the same every time, but usually involves writing music for a game. If there is anything like a game ad/trailer video, then I usually treat that like scoring to picture.

The 2 ai plugins I use are scaler and orb producer. Unfortunately there are no good ai plugins that I have found for use in the entire creation process, but I've found that these 2 are pretty good for brainstorming. Scaler is for chord progressions, and orb producer generates rough musical ideas that are good for building off of. I don't always use them in every project, but they are helpful for when writers block hits!

I couldn't come up with a progression I liked myself, so I used scaler. It has presets the user can scale through, but can also suggest chords. Using the suggest feature in 

**What to look for.** Most participants are broadly positive: they were recruited to talk about how they *use* AI. The interesting variation is in the reservations. Notice how often a drawback is mentioned in passing ("sometimes it's wrong, but...") versus returned to again and again, and notice concerns the starting codebook has no place for. Both observations feed directly into Step 2.

## Step 2 - Adapt the codebook

The codebook below codes two variables per interview:

- `attitude`: the overall stance towards AI in the participant's own work.
- `main_concern`: the **one** concern the participant stresses most.

It is a starting point, not a finished instrument. Compare it with the notes you made while reading. If a definition does not fit what you read, you can change its text (between the quotes) and run the cell again. Otherwise just run it.

Every change is a methodological decision. Keep a short list of what you changed and why.

In [18]:
ATTITUDE = {
    "positive": "sees AI mainly as helpful in their work; reservations are absent or minor asides",
    "mixed":    "describes clear benefits AND at least one reservation they dwell on or return to",
    "negative": "mainly sceptical, reluctant or worried; uses AI little or unwillingly",
}

CONCERN = {
    "job_security":     "fear of losing work, income or clients to AI; devaluation of the profession",
    "skill_loss":       "worry about losing their own skills, learning less, or becoming dependent on AI",
    "accuracy":         "errors, hallucinations, unreliable output, having to check or correct everything",
    "authenticity":     "loss of personal voice, creativity, originality or the human touch in the work",
    "ethics_ip":        "copyright, training data, plagiarism, fairness to other creators",
    "privacy":          "confidential or client data, security",
    "social_judgement": "stigma; colleagues or clients disapproving of AI use; hiding their use",
    "other":            "a clear concern that fits none of the categories above",
    "none":             "the participant expresses no real concern",
}

# Changes to the starting codebook, and why:
# 1. Added `other`. Without it, a concern that fits nowhere is forced into the nearest category.
#    Check how often `other` is used: if it grows large, read those cases and add a category.
# 2. Sharpened `positive` vs `mixed`. Almost every participant mentions *some* drawback, so a
#    passing remark ("sometimes it's wrong, but...") no longer makes an interview `mixed`.
# 3. `accuracy` now includes "having to check or correct", the most common way participants
#    describe the problem in practice.

## Step 3 - Write the prompt and code one interview

For one category per response, a single word was enough in the survey exercise. Here we want three things back from every interview: two codes and a **supporting quote**. The quote is there for you, not for the model. It makes every code checkable: if the model cannot point to where the participant said it, the code is suspect. Models do sometimes invent quotes, so `quote_found()` checks whether the quote really appears in the answers.

`build_prompt()` at the top of the next cell already puts the codebook and the interview into the prompt. **You write the last part: the instruction that tells the model what to return**, which is appended at its end. Write it in plain English. A good instruction:

- asks for a **JSON object** with exactly three keys: `attitude`, `main_concern` and `quote`;
- names the **allowed labels** for each key (copy them from Step 2) and says to use them exactly as written;
- says what the quote must be: copied **word for word** from the answers, short (e.g. at most 25 words), and what to do when there is no concern;
- asks for the JSON object **only**, with nothing before or after it.

Then choose the **decoding settings** (see the *Basics of Text Generation* demo):

- `DO_SAMPLE`: `False` means greedy decoding, so the same interview always gets the same code. `True` samples, so codes can change from run to run. Which one do you want for a coding study?
- `TEMPERATURE`, `TOP_P`, `TOP_K`: only used when `DO_SAMPLE = True` (set them to `None` otherwise). They control how adventurous the sampling is.
- `REPETITION_PENALTY`: `1.0` switches it off. Careful: the penalty also applies to words that appear in the **prompt**, so a high value pushes the model away from the participant's own words, which is exactly what a verbatim quote needs.
- `MAX_NEW_TOKENS`: the room the model has for its reply. The JSON with a 25-word quote needs roughly 80-150 tokens. Too little, and the JSON is cut off and cannot be read (it shows up as a missing code).

In [19]:
def describe(scheme):
    return "\n".join(f"- {label}: {definition}" for label, definition in scheme.items())

SYSTEM = (
    "You are a careful research assistant who codes qualitative interviews with a fixed codebook. "
    "Apply only the categories and definitions you are given, and base every code on what the "
    "participant actually said. Never invent, shorten or paraphrase quotes."
)


def build_prompt(answers):
    """The user message sent for one interview. Your OUTPUT_INSTRUCTION goes at the end."""
    return (
        "Below are one participant's answers from an interview about using AI at work.\n\n"
        f"Variable 1 - attitude towards AI in their own work:\n{describe(ATTITUDE)}\n\n"
        f"Variable 2 - main_concern, the single concern the participant stresses most:\n{describe(CONCERN)}\n\n"
        f'Interview answers:\n"""\n{answers}\n"""\n\n'
        + OUTPUT_INSTRUCTION.strip()
    )


OUTPUT_INSTRUCTION = """
Return a JSON object with exactly these three keys:
  "attitude": one of positive, mixed, negative
  "main_concern": one of job_security, skill_loss, accuracy, authenticity, ethics_ip, privacy, social_judgement, other, none
  "quote": a short quote (at most 25 words) copied word for word from the interview answers that supports main_concern; an empty string "" if main_concern is none
Use the labels exactly as written, in lower case. Copy the quote exactly: do not correct, shorten or join sentences.
Return the JSON object only, with nothing before or after it.
"""

# Coding is measurement: the same interview must always get the same code, so we decode greedily.
DO_SAMPLE = False
TEMPERATURE = None         # ignored when greedy
TOP_P = None               # ignored when greedy
TOP_K = None               # ignored when greedy
REPETITION_PENALTY = 1.0   # off: the quote has to repeat the participant's words, which are in the prompt
MAX_NEW_TOKENS = 200       # a JSON reply with a 25-word quote needs ~80-150 tokens; 200 leaves a margin

In [20]:
require(OUTPUT_INSTRUCTION=OUTPUT_INSTRUCTION.strip(), SYSTEM=SYSTEM, DO_SAMPLE=DO_SAMPLE,
        TEMPERATURE=TEMPERATURE, TOP_P=TOP_P, TOP_K=TOP_K,
        REPETITION_PENALTY=REPETITION_PENALTY, MAX_NEW_TOKENS=MAX_NEW_TOKENS)

DECODING = dict(do_sample=DO_SAMPLE, repetition_penalty=REPETITION_PENALTY, max_new_tokens=MAX_NEW_TOKENS)
if DO_SAMPLE:
    DECODING.update(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K)
print("Decoding:", DECODING)


def code_interview(answers, model=None):
    return parse_json(ask(SYSTEM, build_prompt(answers), json_mode=True, model=model, **DECODING))


def quote_found(quote, answers):
    """True if the quote really appears in the answers (ignoring case, spacing and quote marks)."""
    def norm(s):
        s = str(s).replace("’", "'").replace("“", '"').replace("”", '"')
        return re.sub(r"\s+", " ", s).lower().strip(" .\"'")
    return norm(quote) in norm(answers)


result = code_interview(sample.loc[0, "answers"])
print(json.dumps(result, indent=2, ensure_ascii=False))
print("\nQuote found in the answers:", quote_found(result.get("quote", ""), sample.loc[0, "answers"]))

Decoding: {'do_sample': False, 'repetition_penalty': 1.0, 'max_new_tokens': 200}
{
  "attitude": "mixed",
  "main_concern": "job_security",
  "quote": "the main concern i have is that one day there will not be any more jobs for people like me."
}

Quote found in the answers: True


**What to look for.** The most common problem at this step is a quote that is *almost* right: the model tidies grammar, joins two sentences, or changes "I'm" to "I am". `quote_found()` then returns `False`, and that is correct: it is not what the participant said. If the quote is a clear invention (nothing like it in the answers), treat the whole code with suspicion, not only the quote.

On the decoding settings: with `DO_SAMPLE = True` the same interview can get a different code on every run, so your results would not be reproducible. A high `REPETITION_PENALTY` (e.g. 1.5) is a subtle trap: the participant's words are already in the prompt, so the penalty pushes the model *away* from exactly the words a verbatim quote needs, and more quotes fail. A `MAX_NEW_TOKENS` below about 80 cuts the JSON off, and the interview shows up with missing codes.

Compare the result with your own reading of interview 0. Would you have coded it the same way? If the quote is not found, look at it: is it a paraphrase, a quote stitched together from two places, or an invention? If the reply is empty or a label is wrong, improve `OUTPUT_INSTRUCTION` or the decoding settings (e.g. `MAX_NEW_TOKENS`) and run the two cells again.

## Step 4 - Code the whole sample

This sends one request per interview to the local model. The results are saved to `p1_llm_codes.csv`, so a crashed kernel does not cost you the run.

The checks run automatically. They catch the three silent failures of LLM coding: a label outside the codebook, a reply that was not valid JSON (it shows up as a missing label), and an invented quote.

In [21]:
results = run_parallel(code_interview, sample["answers"].tolist())

coded = sample.join(pd.DataFrame(results, columns=["attitude", "main_concern", "quote"]))
coded["quote_ok"] = [quote_found(q, a) for q, a in zip(coded["quote"], coded["answers"])]
coded.drop(columns=["text"]).to_csv("p1_llm_codes.csv", index=False)

print(f"Attitude outside the codebook : {(~coded['attitude'].isin(ATTITUDE)).sum()} of {len(coded)}")
print(f"Concern outside the codebook  : {(~coded['main_concern'].isin(CONCERN)).sum()} of {len(coded)}")
print(f"Quotes not found in the text  : {(~coded['quote_ok']).sum()} of {len(coded)}")
print()
print(coded["main_concern"].value_counts())

  0%|          | 0/180 [00:00<?, ?it/s]

Attitude outside the codebook : 1 of 180
Concern outside the codebook  : 1 of 180
Quotes not found in the text  : 11 of 180

main_concern
job_security        56
accuracy            56
none                32
authenticity        19
privacy             10
skill_loss           3
social_judgement     2
ethics_ip            1
Name: count, dtype: int64


In [22]:
# Read the interviews that failed. Is there a pattern? If so, improve OUTPUT_INSTRUCTION,
# the codebook or the decoding settings, and run Steps 3 and 4 again.
failed = coded[
    ~coded["attitude"].isin(ATTITUDE)
    | ~coded["main_concern"].isin(CONCERN)
    | ~coded["quote_ok"]
]

print(f"{len(failed)} interviews failed a check\n")
for row in failed.itertuples():
    print(f"{row.transcript_id} ({row.group})  attitude={row.attitude}  main_concern={row.main_concern}")
    print(f"   quote: {row.quote!r}\n")

11 interviews failed a check

creativity_0000 (creatives)  attitude=mixed  main_concern=job_security
   quote: "I'm pretty concerned about how AI generated music might impact streaming platform saturation."

creativity_0108 (creatives)  attitude=mixed  main_concern=authenticity
   quote: "I'm worried that AI will start to generate music and that people will start to lose that personal aspect to it that makes music so wonderful."

creativity_0112 (creatives)  attitude=mixed  main_concern=privacy
   quote: "I am very concerned about AI-generated images being used as reference material when it might contain glaring inaccuracies- imagine I ask for a tattoo design and I don't realize my portrait has six fingers. This might have already happened by now, who knows. Yikes."

creativity_0065 (creatives)  attitude=mixed  main_concern=authenticity
   quote: 'I see AI creating a lot of lackluster creative product and that is one of the reasons that mostly use it for research.'

science_0006 (scien

**What to look for.** Labels outside the codebook are rare with the prefilled `{` and an explicit list of labels. When they occur they are usually near-misses (`job security` with a space, `Mixed`), which you can map back, or a new category the model invented, which you should read. A row with *all* values missing means the reply was not valid JSON: usually `MAX_NEW_TOKENS` was too small and the JSON was cut off. Failed quotes are the most common. If many fail, check first whether they are near-verbatim (punctuation, contractions) or invented. Only the second is a coding problem.

## Step 5 - A second coder from a different model family

Everything so far says how *one* model coded, not whether its codes can be trusted. In a coding study a second coder codes the same material independently, and you report how far the two agree. There is no time to code the interviews by hand today, so the second coder is another LLM.

It should come from a **different model family**. Models of one family (Qwen2.5-1.5B, -3B, -7B, ...) are trained on largely the same data with the same recipe, so they tend to make the same mistakes. Two of them agreeing is like two research assistants trained by the same supervisor agreeing: reassuring, but not independent. Qwen is already taken, so pick something else.

**Find the model on [Hugging Face](https://huggingface.co/models)** (about 10 minutes):

1. Filter by the task **Text Generation** and search for `instruct`. Instruction-tuned models follow a codebook; base models only continue text.
2. Keep it small: **at most about 3B parameters** (shown on the model page, often in the name). Both models have to fit on the GPU together. A T4 has 15 GB, and the first model already uses about 6 GB.
3. Avoid **gated** models: a padlock and "You need to agree to share your contact information to access this model". They need an account, an approved licence request and a login token, which takes longer than you have. Several well-known families, including Llama and Gemma, are gated.
4. Read the model card: the licence, the languages it was trained on, and the context length. The answers run to about 3,000 tokens, so the context has to be at least 4k.
5. Avoid *reasoning* models that write a long `<think>...</think>` section before answering, unless the card shows how to switch that off: the prefilled `{` would be ignored.

Families worth a look: Phi (Microsoft), Granite (IBM), SmolLM (Hugging Face), OLMo (Allen AI), Mistral.

Paste its id (the `organisation/model-name` part of the URL) into `SECOND_MODEL` and run the cell. It codes all 60 interviews with the **same codebook and prompt**; only the model changes.

In [23]:
# Granite 3.3 2B (IBM): Apache-2.0, not gated, 2.5B parameters, 128k context,
# trained on 12 languages. Small enough to sit next to Qwen2.5-3B on a T4.
SECOND_MODEL = "ibm-granite/granite-3.3-2b-instruct"

require(SECOND_MODEL=SECOND_MODEL)

pipe2 = pipeline(
    "text-generation",
    model=SECOND_MODEL,
    dtype="auto",          # on a Colab T4 use dtype=torch.float16
    device_map="auto",
)
print(f"Second coder: {SECOND_MODEL} on {pipe2.model.device} ({pipe2.model.dtype})")

second = run_parallel(lambda answers: code_interview(answers, model=pipe2), coded["answers"].tolist())
second = pd.DataFrame(second, columns=["attitude", "main_concern", "quote"], index=coded.index)

coded["attitude_2"] = second["attitude"]
coded["main_concern_2"] = second["main_concern"]
coded["quote_2"] = second["quote"]
coded["quote_ok_2"] = [quote_found(q, a) for q, a in zip(coded["quote_2"], coded["answers"])]
coded.drop(columns=["text"]).to_csv("p1_llm_codes.csv", index=False)

print(f"Attitude outside the codebook : {(~coded['attitude_2'].isin(ATTITUDE)).sum()} of {len(coded)}")
print(f"Concern outside the codebook  : {(~coded['main_concern_2'].isin(CONCERN)).sum()} of {len(coded)}")
print(f"Quotes not found in the text  : {(~coded['quote_ok_2']).sum()} of {len(coded)}")

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Second coder: ibm-granite/granite-3.3-2b-instruct on cuda:0 (torch.bfloat16)


  0%|          | 0/180 [00:00<?, ?it/s]

Attitude outside the codebook : 0 of 180
Concern outside the codebook  : 0 of 180
Quotes not found in the text  : 41 of 180


**Other good choices** (all open, not gated, at most about 3B parameters): `microsoft/Phi-3.5-mini-instruct` (3.8B, a bit large but fits in float16), `HuggingFaceTB/SmolLM2-1.7B-Instruct` (fast, but weaker at following a long codebook) and `allenai/OLMo-2-0425-1B-Instruct` (fully open training data). If a model fails the format checks on many interviews, look at a few raw replies before comparing, e.g. `ask(SYSTEM, build_prompt(coded.loc[0, "answers"]), json_mode=True, model=pipe2, **DECODING)`.

If the checks above show many labels outside the codebook or many missing values, the second model is struggling with the format rather than the content. If that happens, pick another model and run the cell again.

The next cell compares the two coders. It reports two numbers per variable: the percentage agreement and **Cohen's kappa**. Kappa corrects for the agreement you would get by chance. That matters here: if most participants are `mixed`, two coders who always write `mixed` already agree most of the time.

In [24]:
# Do the two models agree more on attitude or on main_concern? Is kappa much lower
# than the raw agreement, and why?
from sklearn.metrics import cohen_kappa_score

for first, second_col in [("attitude", "attitude_2"), ("main_concern", "main_concern_2")]:
    agreement = (coded[first] == coded[second_col]).mean()
    kappa = cohen_kappa_score(coded[first].astype(str), coded[second_col].astype(str))   # a missing label counts as a disagreement
    print(f"{first:13s} agreement: {agreement:.0%}   kappa: {kappa:.2f}")

disagree = coded[(coded["attitude"] != coded["attitude_2"]) | (coded["main_concern"] != coded["main_concern_2"])]
print(f"\nDISAGREEMENTS: {len(disagree)} of {len(coded)} interviews\n")
print(disagree[["transcript_id", "group", "attitude", "attitude_2", "main_concern", "main_concern_2"]].to_string(index=False))

print("\nConfusion matrix, main_concern (rows: model 1, columns: model 2)")
print(pd.crosstab(coded["main_concern"], coded["main_concern_2"]))

attitude      agreement: 65%   kappa: 0.30
main_concern  agreement: 44%   kappa: 0.27

DISAGREEMENTS: 128 of 180 interviews

  transcript_id      group attitude attitude_2     main_concern main_concern_2
creativity_0042  creatives positive   positive     job_security           none
creativity_0076  creatives positive   positive             none       accuracy
creativity_0053  creatives positive   positive             none       accuracy
creativity_0091  creatives positive      mixed     job_security   job_security
creativity_0056  creatives positive      mixed             none     skill_loss
creativity_0123  creatives positive   positive             none       accuracy
creativity_0093  creatives positive   positive             none   job_security
creativity_0004  creatives    mixed   positive     job_security       accuracy
creativity_0116  creatives    mixed      mixed     job_security   authenticity
creativity_0044  creatives positive      mixed     job_security       accuracy
creati

In [25]:
# Read three disagreements: what the participant said, and each model's code and quote.
for row in disagree.head(3).itertuples():
    print(f"=== {row.transcript_id} ({row.group})")
    print(f"  model 1: {row.attitude} / {row.main_concern}   quote: {row.quote!r}")
    print(f"  model 2: {row.attitude_2} / {row.main_concern_2}   quote: {row.quote_2!r}\n")
    print(row.answers[:2000], "...\n")

=== creativity_0042 (creatives)
  model 1: positive / job_security   quote: 'the barrier to entry might become too low and my clients will turn to AI to do their own editing'
  model 2: positive / none   quote: 'The most recent project that I used AI to help me with was editing out people and vehicle out of an image. Before, doing something like that would have taken an immense amount of time just to remove it, and then you have to worry about replacing it with something to match the background and doing that would just not look seamless because of different lighting and other factors. AI has helped with that process in that it removes, AND adds in background based on the surrounding image and it looks seamless.'

Sounds good to me! No questions, let's begin

My typical creative work is editing real estate photos and videos, every now and then I might have to do some "people" photography and events, but mainly Real Estate.

A typical projects involves going out in person to the propert

Discuss as a team, and write down one sentence for each:

- In the three disagreements you read, **whose code would you defend to a reviewer**: model 1, model 2, or neither? Is it a model error, or a case the codebook does not decide?
- Is agreement higher for `attitude` or for `main_concern`? Why might that be?
- Suppose kappa is high. Does that show the codes are *right*? What might two models from different families still have in common? Before publishing, which interviews would a human have to read, and how many?

**Sample answers**

- **Whose code would you defend?** Expect three kinds of disagreement. One model picks a concern the participant mentions but does not stress (a model error). One model returns a label that is not in the codebook, or no JSON at all (a format error, more common with the smaller second model). Or the participant names two concerns equally and "stresses most" does not decide (a codebook error). Only the third calls for changing the instrument.
- **Attitude vs concern.** Agreement is usually higher for `attitude`: it has 3 categories against 9, and most interviews are clearly positive or mixed. `main_concern` asks for a judgement about *emphasis*, which is where both humans and models disagree. Kappa is usually much lower than raw agreement for `attitude`, because when most interviews are `positive` or `mixed`, two coders agree often by chance alone.
- **Does high kappa show the codes are right?** No. It shows two coders made the same decisions. Models from different families are still trained on overlapping web data, with similar instruction-tuning, and they received the same prompt and codebook, so they can share a bias (e.g. reading polite hedging as a real concern). Inter-model agreement is a cheap way to find the cases worth reading, namely the disagreements. It does not replace a human check: before publishing, a human should code a random sample of about 50 interviews blind, plus all the disagreements.

## Step 6 - The substantive question

Now, and only now, the research question: do the groups differ?

Choose the variable you want to test. The next cell prints row proportions for both variables (the share of each group in each category), then repeats the table for your variable using only the interviews on which both models agree, and tests whether it depends on group. Does the group difference hold when you keep only the interviews both models agree on?

In [26]:
from scipy.stats import chi2_contingency

# attitude: 3 categories give far fewer near-empty cells than main_concern's 9,
# so the chi-square test is less unreliable with low number of interviews per group.
VARIABLE = "attitude"
require(VARIABLE=VARIABLE)

print("ATTITUDE by group (row proportions)")
print(pd.crosstab(coded["group"], coded["attitude"], normalize="index").round(2), "\n")

print("MAIN CONCERN by group (row proportions)")
print(pd.crosstab(coded["group"], coded["main_concern"], normalize="index").round(2), "\n")

both_agree = coded[coded[VARIABLE] == coded[f"{VARIABLE}_2"]]
print(f"{VARIABLE.upper()} by group, only the {len(both_agree)} interviews on which both models agree")
print(pd.crosstab(both_agree["group"], both_agree[VARIABLE], normalize="index").round(2), "\n")

counts = pd.crosstab(coded["group"], coded[VARIABLE])
chi2, p, dof, expected = chi2_contingency(counts)
print(f"Does {VARIABLE} depend on group? chi2 = {chi2:.2f}, df = {dof}, p = {p:.3f}")
print(f"Cells with expected count below 5: {(expected < 5).sum()} of {expected.size} (the test is unreliable if there are many)")

ATTITUDE by group (row proportions)
attitude    mixed  positive
group                      
creatives    0.57      0.43
scientists   0.88      0.12
workforce    0.72      0.28 

MAIN CONCERN by group (row proportions)
main_concern  accuracy  authenticity  ethics_ip  job_security  none  privacy  \
group                                                                          
creatives         0.03          0.20       0.02          0.42  0.23     0.08   
scientists        0.61          0.03       0.00          0.17  0.10     0.05   
workforce         0.30          0.08       0.00          0.35  0.20     0.03   

main_concern  skill_loss  social_judgement  
group                                       
creatives           0.00              0.02  
scientists          0.02              0.02  
workforce           0.03              0.00   

ATTITUDE by group, only the 117 interviews on which both models agree
attitude    mixed  positive
group                      
creatives    0.65      0.35


**Data:** Handa, K. et al. (2025). *Introducing Anthropic Interviewer: What 1,250 professionals told us about working with AI.* https://anthropic.com/research/anthropic-interviewer (data CC-BY).

# Additional ideas

Use OpenAI